In [1]:
from dotenv import load_dotenv
import torch
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from qdrant_client.models import Distance, VectorParams
from scraper import scrape, recipe_id_to_uuid
import asyncio
# import nest_asyncio
# nest_asyncio.apply()


load_dotenv()

True

In [2]:
client = QdrantClient(path="./qdrant_db")
print("✅ Qdrant подключён, папка: ./qdrant_db")

✅ Qdrant подключён, папка: ./qdrant_db


In [3]:
embeddings_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print("✅ Модель загружена")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Модель загружена


In [4]:
COLLECTION_NAME = "recipes"

if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE),
    )
    print(f'✅ Коллекция {COLLECTION_NAME} создана')
else:
    print(f'✅ Коллекция {COLLECTION_NAME} уже существует')

✅ Коллекция recipes создана


In [5]:
vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings_model,
)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

print("✅ VectorStore готов")

✅ VectorStore готов


In [16]:
points, _ = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=10000,
    with_payload=True,
    with_vectors=False
)

points

[Record(id='00000000-0000-0000-0000-000000002f89', payload={'page_content': 'Украинский борщ с пампушками. Категория: Бульоны и супы, Горячие супы, Борщ. Ингредиенты: Бульон, Капуста белокочанная, Свекла, Морковь, Коренья, Перец болгарский, Лук репчатый, Картофель, Чеснок, Фасоль, Сало, Перец черный, Лист лавровый, Зелень, Томатная паста, Сок лимонный, Сметана, Мука пшеничная, Молоко, Дрожжи, Масло растительное, Сахар, Соль, Вода. Назначение: На обед, На первое. Теги: первое, обед. Вкусы: соленый.', 'metadata': {'id': 12169, 'title': 'Украинский борщ с пампушками', 'categories': ['Бульоны и супы', 'Горячие супы', 'Борщ'], 'cuisine': '', 'ingredients': ['Бульон', 'Капуста белокочанная', 'Свекла', 'Морковь', 'Коренья', 'Перец болгарский', 'Лук репчатый', 'Картофель', 'Чеснок', 'Фасоль', 'Сало', 'Перец черный', 'Лист лавровый', 'Зелень', 'Томатная паста', 'Сок лимонный', 'Сметана', 'Мука пшеничная', 'Молоко', 'Дрожжи', 'Масло растительное', 'Сахар', 'Соль', 'Вода'], 'cook_time_minutes': N

In [7]:
from toolkit import RecipeToolkit

recipe_toolkit = RecipeToolkit(vector_store=vector_store)
tools = recipe_toolkit.get_tools()
tools_map = {t.name: t for t in tools}

print("Инструменты:")
for t in tools:
    print(f"  {t.name}: {t.description}")

Инструменты:
  search_recipes: Ищет рецепты в локальной базе данных по смыслу запроса пользователя. 
Используй этот инструмент ПЕРВЫМ при любом запросе о еде или рецептах. 
Поддерживает фильтры по времени готовки, калориям и аллергенам.
  find_similar: Находит похожие рецепты по ID существующего рецепта. 
Используй когда пользователь хочет найти что-то похожее на конкретный рецепт 
и у тебя уже есть его ID из предыдущего поиска.
  scrape_and_save_recipe: Ищет рецепты на povarenok.ru, парсит и сохраняет в базу данных. 
Используй ТОЛЬКО если search_recipes не нашёл подходящих результатов. 
Принимает название блюда транслитом: 'borsch', 'plov', 'pelmeni'.


In [8]:
tools_map

{'search_recipes': StructuredTool(name='search_recipes', description='Ищет рецепты в локальной базе данных по смыслу запроса пользователя. \nИспользуй этот инструмент ПЕРВЫМ при любом запросе о еде или рецептах. \nПоддерживает фильтры по времени готовки, калориям и аллергенам.', args_schema=<class 'toolkit.SearchRecipesInput'>, func=<function RecipeToolkit.get_tools.<locals>.search_recipes at 0x126a35a60>),
 'find_similar': StructuredTool(name='find_similar', description='Находит похожие рецепты по ID существующего рецепта. \nИспользуй когда пользователь хочет найти что-то похожее на конкретный рецепт \nи у тебя уже есть его ID из предыдущего поиска.', args_schema=<class 'toolkit.FindSimilarInput'>, func=<function RecipeToolkit.get_tools.<locals>.find_similar at 0x126afc880>),
 'scrape_and_save_recipe': StructuredTool(name='scrape_and_save_recipe', description="Ищет рецепты на povarenok.ru, парсит и сохраняет в базу данных. \nИспользуй ТОЛЬКО если search_recipes не нашёл подходящих рез

In [9]:
await scrape("borsch", vector_store, limit=10)
# await scrape("plov", vector_store, limit=10)

🔍 Ищем: 'borsch'
   Найдено ссылок: 10
  ✅ Самый красный борщ
  ✅ Пампушки к борщу
  ✅ Борщ
  ✅ Борщ от Пал Саныча
  ✅ Борщ консервированный "Главное-париться не надо!"
  ✅ Пампушки к борщу за 20 минут
  ✅ Украинский борщ от тети Нюси
  ✅ Борщ "Дивской"
  ✅ Борщ по-петровски
  ✅ Украинский борщ с пампушками
✅ Сохранено: 10


[{'id': 87699,
  'title': 'Пампушки к борщу за 20 минут',
  'categories': ['Выпечка', 'Изделия из теста', 'Булочки'],
  'cuisine': '',
  'ingredients': ['Вода',
   'Сахар',
   'Масло подсолнечное',
   'Соль',
   'Дрожжи',
   'Ванильный сахар',
   'Мука пшеничная'],
  'cook_time_minutes': 20,
  'per100_kcal': 273.0,
  'per100_protein': 6.1,
  'per100_fat': 7.0,
  'per100_carbs': 47.3,
  'destiny': ['Для детей',
   'На завтрак',
   'На обед',
   'На полдник',
   'На праздничный стол',
   'На природу',
   'На ужин',
   'Неожиданные гости',
   'Специальное питание',
   'Вегетарианское питание',
   'Постное питание'],
  'tags': ['ужин', 'обед', 'завтрак', 'выпечка', 'пост'],
  'tastes': ['нежный', 'сдобный'],
  'url': 'https://www.povarenok.ru/recipes/show/87699/'},
 {'id': 25436,
  'title': 'Борщ',
  'categories': ['Бульоны и супы', 'Горячие супы', 'Борщ'],
  'cuisine': '',
  'ingredients': ['Мясо',
   'Картофель',
   'Свекла',
   'Морковь',
   'Капуста белокочанная',
   'Помидор',
   'Укс

In [10]:
# Тест 1 — поиск
print(tools_map["search_recipes"].invoke({"query": "борщ"}))

=== Борщ от Пал Саныча ===
ID: 79370
Категории: Бульоны и супы, Горячие супы, Борщ
Время: 240 мин
Ингредиенты: Капуста белокочанная, Свекла, Говядина, Фасоль, Морковь, Лук красный, Картофель, Вода
Назначение: Для детей, На обед, На обед, На первое
Вкусы: овощной, мясной
Калории (100г): 131.4 ккал
Ссылка: https://www.povarenok.ru/recipes/show/79370/

=== Борщ ===
ID: 25436
Категории: Бульоны и супы, Горячие супы, Борщ
Время: —
Ингредиенты: Мясо, Картофель, Свекла, Морковь, Капуста белокочанная, Помидор, Уксус, Сахар, Соль, Укроп, Чеснок
Назначение: Для детей, На обед, На обед, На первое
Вкусы: овощной, насыщенный
Калории (100г): 64.5 ккал
Ссылка: https://www.povarenok.ru/recipes/show/25436/

=== Борщ "Дивской" ===
ID: 66321
Категории: Бульоны и супы, Горячие супы, Борщ
Время: —
Ингредиенты: Лист лавровый, Капуста белокочанная, Картофель, Лук репчатый, Морковь, Масло растительное, Соль, Петрушка, Консервы рыбные, Вода, Перец болгарский
Назначение: Для детей, На обед, Конкурсные рецепты, 

In [11]:
# Тест 2 — с фильтром по времени
print(tools_map["search_recipes"].invoke({"query": "суп", "max_cook_time": 60}))

=== Украинский борщ от тети Нюси ===
ID: 96329
Категории: Бульоны и супы, Горячие супы, Борщ
Время: 60 мин
Ингредиенты: Вода, Свинина, Картофель, Морковь, Свекла, Лук репчатый, Капуста белокочанная, Сало, Масло подсолнечное, Зелень, Помидор, Перец болгарский, Чеснок
Назначение: На обед, На первое
Вкусы: мясной
Калории (100г): 80.2 ккал
Ссылка: https://www.povarenok.ru/recipes/show/96329/

=== Пампушки к борщу за 20 минут ===
ID: 87699
Категории: Выпечка, Изделия из теста, Булочки
Время: 20 мин
Ингредиенты: Вода, Сахар, Масло подсолнечное, Соль, Дрожжи, Ванильный сахар, Мука пшеничная
Назначение: Для детей, На завтрак, На обед, На полдник, На праздничный стол, На природу, На ужин, Неожиданные гости, Специальное питание, Вегетарианское питание, Постное питание
Вкусы: нежный, сдобный
Калории (100г): 273.0 ккал
Ссылка: https://www.povarenok.ru/recipes/show/87699/



In [12]:
# Тест 3 — с фильтром по калориям
print(tools_map["search_recipes"].invoke({"query": "что-нибудь лёгкое", "max_calories": 100.0}))

=== Борщ консервированный "Главное-париться не надо!" ===
ID: 112697
Категории: Заготовки, Суп на зиму
Время: 210 мин
Ингредиенты: Капуста белокочанная, Свекла, Помидор, Перец болгарский, Морковь, Лук репчатый, Кетчуп, Масло подсолнечное, Уксус, Сахар, Соль
Назначение: На обед, На первое, Специальное питание, Вегетарианское питание, Постное питание
Вкусы: овощной, насыщенный, бесподобный
Калории (100г): 97.4 ккал
Ссылка: https://www.povarenok.ru/recipes/show/112697/

=== Украинский борщ от тети Нюси ===
ID: 96329
Категории: Бульоны и супы, Горячие супы, Борщ
Время: 60 мин
Ингредиенты: Вода, Свинина, Картофель, Морковь, Свекла, Лук репчатый, Капуста белокочанная, Сало, Масло подсолнечное, Зелень, Помидор, Перец болгарский, Чеснок
Назначение: На обед, На первое
Вкусы: мясной
Калории (100г): 80.2 ккал
Ссылка: https://www.povarenok.ru/recipes/show/96329/

=== Борщ ===
ID: 25436
Категории: Бульоны и супы, Горячие супы, Борщ
Время: —
Ингредиенты: Мясо, Картофель, Свекла, Морковь, Капуста бел

In [13]:
print(tools_map["search_recipes"].invoke({"query": "борщ", "exclude_ingredients": ["свинина", "сало"]}))

=== Борщ от Пал Саныча ===
ID: 79370
Категории: Бульоны и супы, Горячие супы, Борщ
Время: 240 мин
Ингредиенты: Капуста белокочанная, Свекла, Говядина, Фасоль, Морковь, Лук красный, Картофель, Вода
Назначение: Для детей, На обед, На обед, На первое
Вкусы: овощной, мясной
Калории (100г): 131.4 ккал
Ссылка: https://www.povarenok.ru/recipes/show/79370/

=== Борщ ===
ID: 25436
Категории: Бульоны и супы, Горячие супы, Борщ
Время: —
Ингредиенты: Мясо, Картофель, Свекла, Морковь, Капуста белокочанная, Помидор, Уксус, Сахар, Соль, Укроп, Чеснок
Назначение: Для детей, На обед, На обед, На первое
Вкусы: овощной, насыщенный
Калории (100г): 64.5 ккал
Ссылка: https://www.povarenok.ru/recipes/show/25436/

=== Борщ "Дивской" ===
ID: 66321
Категории: Бульоны и супы, Горячие супы, Борщ
Время: —
Ингредиенты: Лист лавровый, Капуста белокочанная, Картофель, Лук репчатый, Морковь, Масло растительное, Соль, Петрушка, Консервы рыбные, Вода, Перец болгарский
Назначение: Для детей, На обед, Конкурсные рецепты, 

In [15]:
# Тест 5 — find_similar
print(tools_map["find_similar"].invoke({"recipe_id": 66321}))

=== Борщ по-петровски ===
ID: 123061
Категории: Бульоны и супы, Горячие супы, Борщ
Время: 360 мин
Вкусы: богатый, насыщенный
Ссылка: https://www.povarenok.ru/recipes/show/123061/

=== Борщ ===
ID: 25436
Категории: Бульоны и супы, Горячие супы, Борщ
Время: —
Вкусы: овощной, насыщенный
Ссылка: https://www.povarenok.ru/recipes/show/25436/

=== Борщ от Пал Саныча ===
ID: 79370
Категории: Бульоны и супы, Горячие супы, Борщ
Время: 240 мин
Вкусы: овощной, мясной
Ссылка: https://www.povarenok.ru/recipes/show/79370/

=== Украинский борщ с пампушками ===
ID: 12169
Категории: Бульоны и супы, Горячие супы, Борщ
Время: —
Вкусы: соленый
Ссылка: https://www.povarenok.ru/recipes/show/12169/

